Install packages

In [ ]:
!pip -q install datasets pandas scikit-learn pyarrow

Mount Drive and create folders

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

BASE_PATH = Path("/content/drive/MyDrive/AI-Resume-Screener")

for folder in [
    "data/raw",
    "data/processed",
    "models/word2vec",
    "models/bert",
    "outputs",
]:
    (BASE_PATH / folder).mkdir(parents=True, exist_ok=True)

print(BASE_PATH)

Mounted at /content/drive
/content/drive/MyDrive/AI-Resume-Screener


Load the paired resume-job dataset

In [ ]:
from datasets import load_dataset

fit_dataset = load_dataset("med2425/resume-job-fit-merged-v1")
fit_dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/2.86k [00:00<?, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/129M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/128M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/43.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/80017 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/13716 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['resume', 'jd', 'label', 'source', 'resume_domain', 'jd_domain'],
        num_rows: 80017
    })
    test: Dataset({
        features: ['resume', 'jd', 'label', 'source', 'resume_domain', 'jd_domain'],
        num_rows: 13716
    })
})

Convert to pandas and inspect

In [ ]:
import pandas as pd

train_df = fit_dataset["train"].to_pandas()
test_df = fit_dataset["test"].to_pandas()

print(train_df.shape, test_df.shape)
display(train_df.head(2))
print(train_df["label"].value_counts())

(80017, 6) (13716, 6)


,resume,jd,label,source,resume_domain,jd_domain
0,SummarySelf-motivated accountant offering a s...,"Our client, a Raleigh-based full-service comme...",Potential Fit,generated_smart,finance,finance
1,Professional ProfileSpecialized knowledge of r...,"""All candidates must be directly contracted by...",No Fit,generated_smart,finance,software


label
No Fit           60313
Potential Fit    13907
Good Fit          5797
Name: count, dtype: int64


Clean invalid rows and convert labels to 0-1 scores

In [ ]:
LABEL_TO_SCORE = {
    "No Fit": 0.0,
    "Potential Fit": 0.5,
    "Good Fit": 1.0,
}

required_columns = ["resume", "jd", "label"]

train_df = train_df.dropna(subset=required_columns).copy()
test_df = test_df.dropna(subset=required_columns).copy()

train_df = train_df[train_df["label"].isin(LABEL_TO_SCORE)].copy()
test_df = test_df[test_df["label"].isin(LABEL_TO_SCORE)].copy()

train_df["score"] = train_df["label"].map(LABEL_TO_SCORE).astype(float)
test_df["score"] = test_df["label"].map(LABEL_TO_SCORE).astype(float)

train_df = train_df.drop_duplicates(subset=["resume", "jd"]).reset_index(drop=True)
test_df = test_df.drop_duplicates(subset=["resume", "jd"]).reset_index(drop=True)

print(train_df.shape, test_df.shape)

(80017, 7) (13716, 7)


Create validation set

In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    train_df,
    test_size=0.10,
    random_state=42,
    stratify=train_df["label"],
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (72015, 7)
Validation: (8002, 7)
Test: (13716, 7)


Save processed splits

In [ ]:
train_df.to_parquet(BASE_PATH / "data/processed/train.parquet", index=False)
val_df.to_parquet(BASE_PATH / "data/processed/validation.parquet", index=False)
test_df.to_parquet(BASE_PATH / "data/processed/test.parquet", index=False)

print("Saved all splits.")

Saved all splits.
